### Coleta de dados sobre o IDHM - Índice de Desenvolvimento Humano - Muncipal

<pre>
Fonte:
    https://www.undp.org/sites/g/files/zskgke326/files/2023-07/base_de_dados.xlsx

Dados a serem extraídos:
Sigla	Categoria	Significado / Descrição
IDHM	Índices Gerais	Índice de Desenvolvimento Humano Municipal (média geométrica de Longevidade, Educação e Renda).
IDHM_L	Índices Gerais	IDHM - Dimensão Longevidade (saúde e expectativa de vida).
IDHM_E	Índices Gerais	IDHM - Dimensão Educação (frequência escolar e escolaridade adulta).
IDHM_R	Índices Gerais	IDHM - Dimensão Renda (padrão de vida e capacidade de consumo).    

</pre>

In [19]:
import sys, os
import requests
import openpyxl
import pandas as pd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\mais_einstein\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
# url_pnud_idhm = "https://www.undp.org/sites/g/files/zskgke326/files/2023-07/base_de_dados.xlsx"


url = "https://www.undp.org/sites/g/files/zskgke326/files/2023-07/base_de_dados.xlsx"
local_file = "base_de_dados.xlsx"

# Download
response = requests.get(url)

r = requests.get(url)

print("Status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type"))
print("Tamanho:", len(r.content))

# Status: 403
# Content-Type: text/html
# Tamanho: 462

# Acesso bloqueado, arquivo baixado manualmente

with open(local_file, 'wb') as f:
    f.write(response.content)



In [ ]:
# Leitura com Pandas

file = r"C:\Marco Conti\Projetos\mais_einstein\IDHM\base_de_dados.xlsx"

df_pandas = \
    pd.read_excel(
        file,
        sheet_name="Base de Dados",
        engine="openpyxl"
    )


# Conversão para Spark
# spark = SparkSession.builder.appName("UNDPData").getOrCreate()
df_spark = spark.createDataFrame(df_pandas)
df_spark.show()

In [25]:
df_spark.createOrReplaceTempView("temp_pnud_idhm")
df_spark.printSchema()

query = """Select agregacao
                 ,codigo
                 ,nome
                 ,idhm
                 ,idhm_L as idhm_longevidade
                 ,idhm_E as idhm_educacao
                 ,idhm_R as idhm_renda
             from temp_pnud_idhm
            where ano = 2021
        """

spark.sql(query).show()


# (df_spark
#     .filter("AGREGACAO = 'UF'")
#     .select("ANO", "AGREGACAO", "CODIGO")
#     .groupBy("ANO", "AGREGACAO").agg(F.max("CODIGO"))
#     .show(10,False))

root
 |-- ANO: long (nullable = true)
 |-- AGREGACAO: string (nullable = true)
 |-- CODIGO: double (nullable = true)
 |-- NOME: string (nullable = true)
 |-- IDHM: double (nullable = true)
 |-- IDHM_L: double (nullable = true)
 |-- IDHM_E: double (nullable = true)
 |-- IDHM_R: double (nullable = true)
 |-- IDHMAD: double (nullable = true)
 |-- IDHMAD_L: double (nullable = true)
 |-- IDHMAD_E: double (nullable = true)
 |-- IDHMAD_R: double (nullable = true)
 |-- IDHMAD_PERDA: double (nullable = true)
 |-- IDHMAD_L_PERDA: double (nullable = true)
 |-- IDHMAD_E_PERDA: double (nullable = true)
 |-- IDHMAD_R_PERDA: double (nullable = true)
 |-- ESPVIDA: double (nullable = true)
 |-- T_FREQ5A6: double (nullable = true)
 |-- T_FUND11A13: double (nullable = true)
 |-- T_FUND15A17: double (nullable = true)
 |-- T_MED18A20: double (nullable = true)
 |-- T_FUND18M: double (nullable = true)
 |-- ANOSEST: double (nullable = true)
 |-- RDPC: double (nullable = true)
 |-- GINI: double (nullable = tru